### Imports

In [ ]:
import pandas as pd
import os
import kagglehub
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
dataset_name = 'rmisra/news-headlines-dataset-for-sarcasm-detection'
file_name = 'Sarcasm_Headlines_Dataset_v2.json'

### Data Processing

In [2]:
path = kagglehub.dataset_download(dataset_name)
data_path = os.path.join(path, file_name)
df = pd.read_json(data_path, lines=True)

### Parsing

In [3]:
all_words = set()

for headline in df['headline']:
  words = headline.lower().split()
  for word in words:
    all_words.add(word)

all_words = list(all_words)
n_words = len(all_words)

In [4]:
headlines = []
word_dict = {}

for i, word in enumerate(all_words):
  word_dict[word] = i + 2

for headline in df['headline']:
  words = headline.lower().split()
  indices = [word_dict.get(word, 1) for word in words]
  headline_tensor = torch.tensor(indices)
  headlines.append(headline_tensor)

### Encoding

In [5]:
padded_headlines = pad_sequence(headlines, batch_first=True, padding_value=0) # adds padding and unknown
labels = torch.tensor(df['is_sarcastic'].values, dtype=torch.float32)
dataset = TensorDataset(padded_headlines, labels)

train_size = int(len(dataset) * 0.85)
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=64, shuffle=True)

### The Network

In [ ]:
class SarcasmRNN(nn.Module):
  def __init__(self, n_words, embed_dim=128, hidden_dim=256):
    super().__init__()

    self.embedding = nn.Embedding(n_words, embed_dim, padding_idx=0)
    self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, 1)
  
  def forward(self, x):
    embedded = self.embedding(x)
    out, hidden = self.rnn(embedded)
    final = self.fc(hidden[-1])
    return final.squeeze(1)